# 3.3. Фракталы как вычислительный эксперимент


Рассматриваем $z_{n+1}=z_n^2+c$. Для множества Мандельброта варьируется $c$ при $z_0=0$

Для заполненного множества Джулиа $K_c$ варьируется $z_0$ при фиксированном $c$

множество Джулиа $J_c$ — граница $K_c$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import display, Image

cmap = plt.get_cmap("turbo").copy()
cmap.set_bad("black")

mandel_bounds = (-2, 1, -1.5, 1.5)
julia_bounds = (-1.8, 1.8, -1.8, 1.8)
zoom_bounds = (-0.748, -0.742, 0.128, 0.134)
parameters = [0, -1, -0.123 + 0.745j, 1]

active — маска точек, которые ещё не вышли за радиус. Для Мандельброта берём $R=2$,
для Джулиа — $R=\max(2,|c|)$.

Выход проверяем по строгому неравенству $|z|>R$.

Номер выхода записываем в first_exit, а если выхода не было, оставляем -1.

In [ ]:
def complex_grid(bounds, size):
    xmin, xmax, ymin, ymax = bounds
    x = xmin + (np.arange(size) + 0.5) * (xmax - xmin) / size
    y = ymin + (np.arange(size) + 0.5) * (ymax - ymin) / size
    return x[None, :] + 1j * y[:, None]


def escape_time(z0, c, max_iter=200, radius=2):
    z, c = np.broadcast_arrays(np.asarray(z0, dtype=complex), np.asarray(c, dtype=complex))
    z = z.copy()
    active = np.ones(z.shape, dtype=bool)
    first_exit = np.full(z.shape, -1, dtype=int)
    smooth = np.full(z.shape, np.nan)

    for n in range(max_iter + 1):
        # При n=0 проверяем исходные точки, не делая шага отображения.
        modulus = np.abs(z)
        escaped = active & (modulus > radius)
        first_exit[escaped] = n
        smooth[escaped] = n - np.log2(np.log(modulus[escaped]) / np.log(radius))
        active[escaped] = False
        if n == max_iter or not active.any():
            break
        z[active] = z[active] ** 2 + c[active]

    return first_exit, smooth


def mandelbrot(bounds, size, max_iter):
    return escape_time(0, complex_grid(bounds, size), max_iter, radius=2)


def julia(c, size=600, max_iter=200):
    return escape_time(complex_grid(julia_bounds, size), c, max_iter, radius=max(2, abs(c)))


def draw_map(ax, values, bounds, title, norm, plane="z"):
    masked = np.ma.masked_invalid(values.astype(float))
    picture = ax.imshow(masked, origin="lower", extent=bounds, cmap=cmap, norm=norm, interpolation="nearest")
    ax.set(title=title, xlabel=f"Re {plane}", ylabel=f"Im {plane}")
    ax.set_aspect("equal")
    return picture


def exit_colors(counts):
    return np.where(counts < 0, np.nan, counts)

## Мандельброт и Джулиа

Сетка $600\times600$, 200 итераций. Цвет показывает номер выхода, чёрный — выход не найден.
Шкала `PowerNorm` растягивает область малых номеров, чтобы лучше видеть рисунок.

In [ ]:
mandel_counts, mandel_smooth = mandelbrot(mandel_bounds, 600, 200)
julia_results = {c: julia(c) for c in parameters}
norm200 = PowerNorm(gamma=0.3, vmin=0, vmax=200)

fig, ax = plt.subplots(figsize=(7.5, 6), layout="constrained")
im = draw_map(ax, exit_colors(mandel_counts), mandel_bounds, "Мандельброт: 600 на 600, Nmax = 200", norm200, plane="c")

for i, c in enumerate(parameters, 1):
    ax.plot(np.real(c), np.imag(c), "wo", mec="black", ms=6, clip_on=False)
    ax.annotate(f"{i}: c = {c}", (np.real(c), np.imag(c)), xytext=(-55, -25) if c == 1 else (8, 10), textcoords="offset points", color="white", bbox=dict(facecolor="black", alpha=0.7, pad=2))
fig.colorbar(im, ax=ax, label="Номер выхода n", ticks=[0, 1, 5, 20, 50, 100, 200])
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(9, 8), layout="constrained")
for i, (ax, c) in enumerate(zip(axes.flat, parameters), 1):
    counts, _ = julia_results[c]
    im = draw_map(ax, exit_colors(counts), julia_bounds, f"{i}: c = {c}", norm200)
    if c == 0:
        theta = np.linspace(0, 2 * np.pi, 500)
        ax.plot(np.cos(theta), np.sin(theta), "w--", lw=1, label="|z| = 1")
        ax.legend(loc="upper right")

fig.suptitle("Заполненные множества Джулиа: 600 на 600, Nmax = 200")
fig.colorbar(im, ax=axes.ravel().tolist(), label="Номер выхода n", ticks=[0, 1, 5, 20, 50, 100, 200], shrink=0.8)

plt.show()

При $c=0$ получаем $|z_n|=|z_0|^{2^n}$. Поэтому заполненное множество — единичный диск,
а его граница — единичная окружность. Проверим совпадение на сетке.

In [ ]:
grid = complex_grid(julia_bounds, 600)
disk = np.abs(grid) <= 1
not_escaped = julia_results[0][0] == -1
print("Несовпадений с единичным диском:", np.count_nonzero(disk != not_escaped))

## Увеличение

Возьмём участок $\operatorname{Re}c\in[-0.748,-0.742]$, $\operatorname{Im}c\in[0.128,0.134]$.

Каждый раз считаем его заново. По горизонтали сравниваем сетки, по вертикали - число итераций.

Цветовая шкала у всех четырёх рисунков одинаковая.

In [ ]:
zoom_results = {}
for max_iter in [200, 800]:
    for size in [600, 1200]:
        zoom_results[size, max_iter] = mandelbrot(zoom_bounds, size, max_iter)

norm800 = PowerNorm(gamma=0.3, vmin=0, vmax=800)
fig, axes = plt.subplots(2, 2, figsize=(10, 9), layout="constrained")

for row, max_iter in enumerate([200, 800]):
    for col, size in enumerate([600, 1200]):
        counts, _ = zoom_results[size, max_iter]
        im = draw_map(axes[row, col], exit_colors(counts), zoom_bounds, f"{size} на {size}, Nmax = {max_iter}", norm800, plane="c")
        axes[row, col].ticklabel_format(useOffset=False)

fig.colorbar(im, ax=axes.ravel().tolist(), label="Номер выхода n", ticks=[0, 5, 20, 50, 200, 400, 800], shrink=0.8)
plt.show()

for size in [600, 1200]:
    n200 = zoom_results[size, 200][0]
    n800 = zoom_results[size, 800][0]
    later_exit = (n200 == -1) & (n800 >= 0)
    print(f"Сетка {size}: выход на шагах 201–800 у {100 * later_exit.mean():.2f}% точек")

## Плавная окраска

Вместо целого номера выхода используем
$$\nu=n-\log_2\left(\frac{\ln|z_n|}{\ln R}\right).$$
Поправка учитывает, насколько далеко точка вышла за радиус. Она получается из приближения
$|z_{n+s}|\approx|z_n|^{2^s}$, если найти дробный момент достижения $R$.
Полосы становятся менее заметными, хотя формула приближённая.
Сравним окраски на небольшом внешнем участке: сетка $600\times600$, 800 итераций.

In [ ]:
color_bounds = (-0.7439, -0.7435, 0.1296, 0.1300)
counts, smooth = mandelbrot(color_bounds, 600, 800)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), layout="constrained")
color_norm = PowerNorm(gamma=1, vmin=51, vmax=55)
for ax, values, title in zip(axes, [exit_colors(counts), smooth], ["Целый номер n", "Дробный индекс ν"]):
    im = draw_map(ax, values, color_bounds, title, color_norm, plane="c")
    ax.ticklabel_format(useOffset=False)
    
fig.colorbar(im, ax=axes.tolist(), label="Индекс окраски (общая шкала)")
plt.show()

## Анимация

Пусть $c$ движется по действительной оси от $-0.8$ до $0.4$.
Границу Мандельброта этот отрезок пересекает при $c=0.25$.
Для нескольких кадров проверим критическую орбиту из $z_0=0$ на 800 шагах.

In [ ]:
c_path = np.linspace(-0.8, 0.4, 81)
for frame in [0, 40, 68, 70, 72, 80]:
    c = c_path[frame]
    counts, _ = escape_time(0, c, max_iter=800)
    n = counts.item()
    result = "выход не найден" if n == -1 else f"выход на шаге {n}"
    print(f"Кадр {frame}, c = {c:.2f}: {result}")

При $c=0.28$ выход происходит на 16-м шаге, при $c=0.4$ — на 7-м.
Для остальных выбранных значений выхода нет. Их ограниченность здесь подтверждается
известным свойством действительного отрезка $[-2,1/4]$, а не только расчётом.

Слева в анимации отмечен параметр, справа — соответствующее заполненное множество Джулиа.
Для скорости берём сетку $300\times300$ и 200 итераций.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2), layout="constrained")
draw_map(axes[0], mandel_smooth, mandel_bounds, "Параметр на карте Мандельброта", norm200, plane="c")
axes[0].plot(c_path, np.zeros_like(c_path), "w-", lw=1)
axes[0].plot(0.25, 0, "w+", ms=10)
axes[0].annotate("c = 1/4", (0.25, 0), xytext=(-0.1, 0.35), color="white", arrowprops=dict(arrowstyle="->", color="white"))
marker, = axes[0].plot([], [], "o", color="red", mec="white", ms=7)
_, first_smooth = julia(c_path[0], size=300, max_iter=200)
im = draw_map(axes[1], first_smooth, julia_bounds, "", norm200)
fig.colorbar(im, ax=axes.tolist(), label="Плавный индекс ν; чёрный — нет выхода", ticks=[0, 1, 5, 20, 50, 200], shrink=0.8)


def update(frame):
    c = c_path[frame]
    _, values = julia(c, size=300, max_iter=200)
    marker.set_data([c], [0])
    im.set_data(np.ma.masked_invalid(values))
    axes[1].set_title(f"Kc: c = {c:.3f}; 300 на 300, Nmax = 200")
    return marker, im


animation = FuncAnimation(fig, update, frames=len(c_path), interval=100, blit=False)
gif_path = "julia_parameter.gif"
animation.save(gif_path, writer=PillowWriter(fps=10), dpi=90)
plt.close(fig)
display(Image(filename=gif_path))

Если критическая орбита ограничена, множество Джулиа связно. Если она уходит,
множество вполне несвязно.